# Module 0 — Data Pipeline & the ML Workflow
**Lectures:** Week 1 (ML workflow, features, regression vs classification) · Week 2 (tools, X/y, shape, train/test split)

**Real-world problem:** Ontario's grid operator (IESO) must forecast hourly demand a day ahead. Under-forecast → blackouts / expensive peaker plants. Over-forecast → wasted reserve. Every module builds a piece of that forecaster.

**Goal today:** pull live data, understand every column, and build `X`, `y` correctly.

## 1. Pull live data
Run once in a terminal from the repo root (re-run any time to refresh):
```bash
python scripts/update_data.py
```
It downloads IESO hourly demand + Toronto weather and saves `data/processed/dataset.parquet` (git-ignored — data is rebuilt, not committed).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.describe().T[['mean','min','max']]

## 2. Look at the data (Week 1: *visualise before modelling*)
One row = one hour. Target candidates:
- `demand_mw` → **regression** (predict a number)
- `is_peak` (we create it) → **classification** (predict a class)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
last2w = df.tail(24 * 14)
ax[0].plot(last2w["timestamp_utc"], last2w["demand_mw"]); ax[0].set_title("Last 2 weeks: daily cycle")
ax[1].scatter(df["temperature_2m"], df["demand_mw"], s=2, alpha=0.3)
ax[1].set_xlabel("Temperature (°C)"); ax[1].set_ylabel("Demand (MW)"); ax[1].set_title("Demand vs temperature")
plt.tight_layout()

**Predict before you look closely:** is demand vs temperature a straight line? Write your answer here, then check the plot. (This shape is the whole story of Module 4.)

> Your answer:

## 3. Build X and y (Week 2 notation)
`X` = feature matrix, shape `(n_samples, n_features)`. `y` = target vector, shape `(n_samples,)`.

In [ ]:
FEATURES = ["hour", "dayofweek", "month", "is_weekend", "is_holiday", "temperature_2m"]

# TODO 1: build X (only the FEATURES columns) and y (the demand_mw column)
X = None
y = None

# TODO 2: split 80/20 with train_test_split, random_state=42
X_train, X_test, y_train, y_test = None, None, None, None

In [ ]:
# Checks — these should all pass
assert X.shape == (len(df), len(FEATURES))
assert y.ndim == 1
assert len(X_train) + len(X_test) == len(df)
print("X_train", X_train.shape, "| X_test", X_test.shape)

## 4. Why questions (answer in your own words)
1. Why is `X` 2-D but `y` 1-D?
2. Which columns could you **not** use to forecast *tomorrow's* demand, because you wouldn't know them yet?
3. `demand_lag_24h` is in the data. Is it legal for a day-ahead forecast? Why?

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-00-data          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 0: data pipeline + X/y split"
git push -u origin module-00-data
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.
_First module: see `docs/GIT_TRACK.md` Stage 1 to create the repo and push `main` first._